# Page 6: what happens when a model runs out of tokens

A model reasons by writing its chain of thought one token at a time. This notebook gives three
Qwen2.5 models a hard limit on how many tokens they may write, from 128 up to 2,048, and measures
how many of 100 hard maths problems each one still solves.

The token budget is the only thing that changes. Every run uses the same 100 problems (MATH-500,
levels 4 and 5, the same set as page 5), the same prompt, and greedy decoding, so a model given the
same budget always writes the same chain.

**What it writes**, in `out/page_06/`:

| File | What it holds |
|---|---|
| `traces.jsonl` | every chain the models wrote, token by token |
| `derived.json` | the numbers the website draws |
| `manifest.json` | how, where and with what the run was made |

Shared helpers (model loading, generation from token ids, the resumable log) live in `harness.py`,
which ships beside this notebook.

## 1. How to run it

| Mode | What it does | Hardware | Time |
|---|---|---|---|
| `REPLAY` (default) | Rebuilds every number and figure from the published chains. Nothing is generated. | any laptop | under a minute |
| `QUICK` | Generates a small run: Qwen2.5-3B, 30 problems, 3 budgets. | GPU, ~6 GB | ~20 minutes |
| `FULL` | Generates the published run: 3 models, 100 problems, 5 budgets. | GPU, 16 GB+ | 2 to 3 hours |

Choose the mode in the next cell, or from a shell:

```bash
PAGE6_RUN_MODE=FULL jupyter nbconvert --to notebook --execute page_06_thinking_longer.ipynb
```

Generation is resumable: each chain is appended to `traces.jsonl` as soon as it is written, so an
interrupted run carries on from where it stopped. Install the dependencies with
`pip install -r requirements.txt`.

In [ ]:
import os

RUN_MODE = os.environ.get("PAGE6_RUN_MODE", "REPLAY")      # "REPLAY", "QUICK" or "FULL"

PRIMARY = "Qwen/Qwen2.5-3B-Instruct"        # the model figures 1 and 3 are drawn from
ALL_MODELS = [                              # (model, precision)
    (PRIMARY, "bf16"),
    ("Qwen/Qwen2.5-1.5B-Instruct", "bf16"),
    ("Qwen/Qwen2.5-7B-Instruct", "8bit"),   # 8-bit so that it fits on a 16 GB card
]
MODES = {
    "REPLAY": dict(models=ALL_MODELS, n_items=100, budgets=[128, 256, 512, 1024, 2048], waits=[0, 1, 2, 3]),
    "QUICK":  dict(models=ALL_MODELS[:1], n_items=30, budgets=[128, 512, 2048], waits=[0, 1, 2]),
    "FULL":   dict(models=ALL_MODELS, n_items=100, budgets=[128, 256, 512, 1024, 2048], waits=[0, 1, 2, 3]),
}
CFG = MODES[RUN_MODE]
MODELS, N_ITEMS = CFG["models"], CFG["n_items"]
BUDGETS = sorted(CFG["budgets"])
MAX_BUDGET = BUDGETS[-1]
SEED = 123                  # chooses which problems are used; decoding itself is greedy

# The comparison in which a cut-off model is made to answer anyway (section 3).
ANSWER_ALLOWANCE = 32       # tokens it may write after "Final answer: \boxed{"

# Appendix A: asking for a short answer instead of imposing a cap.
RUN_ASKED_NICELY = True

# Appendix B: the opposite of a cap, telling the model to keep going.
RUN_WAIT_ARM = True
WAIT_WORD = "Wait"
WAITS = sorted(CFG["waits"])     # how many times it is told to keep going
WAIT_ALLOWANCE = 512             # tokens each "Wait" buys

GENERATES = RUN_MODE != "REPLAY"
assert PRIMARY in dict(MODELS), "the primary model must be one of the models in the run"
print(f"mode {RUN_MODE}: {N_ITEMS} problems, budgets {BUDGETS}")
for model_id, quant in MODELS:
    print(f"   {model_id.split('/')[-1]:<24}{quant}")

In [ ]:
import json
import random
import textwrap
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.colors import ListedColormap

import harness
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
    extract_final_candidate,
    get_last_boxed,
    grade_answer,
    load_math500_test,
    render_prompt,
)

OUT_DIR, DATA_DIR = Path("out/page_06"), Path("data")
for folder in (OUT_DIR, DATA_DIR):
    folder.mkdir(parents=True, exist_ok=True)
TRACES_PATH = OUT_DIR / "traces.jsonl"
DERIVED_PATH = OUT_DIR / "derived.json"
MANIFEST_PATH = OUT_DIR / "manifest.json"

mean = harness.mean
print("torch", torch.__version__, "| harness", harness.__version__)

## 2. The problems

100 problems from MATH-500 at levels 4 and 5, the two hardest levels. They are drawn with a fixed
seed and the list is hashed, so "the same problems as page 5" can be checked instead of taken on
trust.

In [ ]:
math500 = load_math500_test(local_path=DATA_DIR / "math500_test.json")
pool = sorted((r for r in math500 if int(r["level"]) in (4, 5)), key=lambda r: r["unique_id"])
random.Random(SEED).shuffle(pool)
items = pool[:N_ITEMS]

ITEM_SET_HASH = harness.sha256_of(r["unique_id"] for r in items)
ORDER = {r["unique_id"]: i for i, r in enumerate(items)}     # every list below keeps this order

levels = Counter(int(r["level"]) for r in items)
print(f"{len(items)} of the {len(pool)} level 4 and 5 problems: {levels[4]} at level 4, "
      f"{levels[5]} at level 5   (item set {ITEM_SET_HASH[:12]})")

## 3. How answers are graded

The grader is the one from *Reasoning from Scratch* (chapter 3), unchanged. It takes the model's
last `\boxed{...}` answer and compares it with the reference, treating equivalent forms such as
`\frac{1}{2}` and `0.5` as equal.

A chain that finishes inside its budget is graded normally. A chain that hits the budget can be
graded two ways, and this notebook reports both:

- **Natural stop (the main result).** The model simply stops. A chain that was cut off counts as
  correct only if it had already written a complete `\boxed{}` answer before the cut. This is what
  actually happens when you cap a model.
- **Forced answer (the comparison).** At the cap we append `Final answer: \boxed{` and let the model
  write up to 32 more tokens. This is the "budget forcing" of Muennighoff et al. (2025): the best
  answer a model can give when it is cut off.

One detail matters for natural stop. When a text contains no `\boxed{}`, the grader's extractor
falls back to the *last number* in the text. A number in the middle of the working is not an
answer, so natural-stop grading looks for a `\boxed{}` answer only.

In [ ]:
harness.check_grader([
    # (what the model wrote, reference answer, should it count as correct?)
    (r"The answer is \boxed{9}.", "9", True),
    (r"27", "9", False),
    (r"\boxed{\frac{1}{2}}", "0.5", True),
    (r"\boxed{1,024}", "1024", True),
    (r"\boxed{\left( 3, \frac{\pi}{2} \right)}", r"\left( 3, \frac{\pi}{2} \right)", True),
    # cut off mid-sentence, then forced to answer
    ("... 3/4 = 9/12 and 2/3 = 8/12. Adding gives 17/12. Now I need to subtr"
     "\n\nFinal answer: " + r"\boxed{\frac{17}{12}}", r"\frac{17}{12}", True),
    # forced, but the forced answer is wrong: forcing must not rescue it
    ("...\n\nFinal answer: " + r"\boxed{\frac{5}{12}}", r"\frac{17}{12}", False),
], extract_final_candidate, grade_answer)


def natural_stop_correct(rec):
    """Grade a record as if the model had simply stopped at its token budget."""
    if not rec["truncated"]:
        return rec["correct"]                          # it finished on its own
    own = "".join(rec["tokens"][: rec["cut_index"]])   # only what it wrote before the cut
    boxed = get_last_boxed(own)
    return bool(boxed) and bool(grade_answer(boxed.strip().strip("$ "), rec["ground_truth"]))


# A cut-off chain with no boxed answer is wrong under natural stop, even though the
# last-number fallback would have picked out "12".
assert not natural_stop_correct(
    dict(truncated=True, cut_index=1, ground_truth="12", tokens=["so 3 x 4 = 12, and", " then"]))

## 4. The models, or the published run

Three instruction-tuned Qwen2.5 models with 1.5B, 3B and 7B parameters. They share one tokenizer,
which is what lets them share a token-budget axis. The 7B is loaded in 8-bit so that it fits on a
16 GB card; its precision is the same at every budget, so it can move that model's curve up or down
but not change its shape.

In `REPLAY` mode nothing is loaded. The next cell reads the published chains instead, and every
cell after it works the same way whichever mode produced them.

In [ ]:
results = {}       # results[model][budget]: one record per problem, in problem order
asked_nicely = []  # appendix A's records
wait_records = {}  # appendix B's records, by model
model_meta = {}    # what was loaded, for the manifest

if RUN_MODE == "REPLAY":
    assert TRACES_PATH.exists() and MANIFEST_PATH.exists(), (
        f"REPLAY reads the published run from {OUT_DIR}/traces.jsonl and manifest.json, "
        f"which come with the notebook download.")
    RUN_MANIFEST = json.loads(MANIFEST_PATH.read_text())
    RUN_ID = RUN_MANIFEST["run_id"]
    assert RUN_MANIFEST["items"]["item_set_sha256"] == ITEM_SET_HASH, \
        "the published run was made on a different set of problems"

    roster = {model_id for model_id, _ in MODELS}
    with TRACES_PATH.open(encoding="utf-8") as f:
        for r in (json.loads(line) for line in f if line.strip()):
            if r["run_id"] != RUN_ID or r["problem_id"] not in ORDER:
                continue
            if r.get("arm") == "wait":
                wait_records.setdefault(r["model_id"], []).append(r)
            elif r["model_id"] == "soft":
                asked_nicely.append(r)
            elif r["model_id"] in roster and r["budget"] in BUDGETS:
                results.setdefault(r["model_id"], {}).setdefault(r["budget"], []).append(r)
    model_meta = {m["id"]: m for m in RUN_MANIFEST["models"]}
    print(f"replaying {RUN_ID}; no model loaded")
else:
    device = get_device()
    DTYPE = harness.pick_dtype(device)
    GPU_NAME = torch.cuda.get_device_name(0) if device.type == "cuda" else device.type
    harness.assert_same_gpu(MANIFEST_PATH, GPU_NAME)
    RUN_ID = (f"{RUN_MODE.lower()}-math500_l45-n{len(items)}-{ITEM_SET_HASH[:8]}"
              f"-cap{MAX_BUDGET}-{str(DTYPE).rsplit('.', 1)[-1]}-{device.type}")
    STORE = harness.TraceStore(TRACES_PATH, RUN_ID, ("model_id", "budget", "problem_id"))
    STORE.report()

    def load(model_id, quant):
        got = harness.load_model(model_id, device=device, dtype=DTYPE,
                                 load_in_8bit=(quant == "8bit"), warmup_tokens=16)
        mdl, tok = got["model"], got["tokenizer"]
        model_meta[model_id] = dict(
            id=model_id, label=model_id.split("/")[-1], params=got["params"],
            revision=harness.repo_revision(model_id), dtype=str(next(mdl.parameters()).dtype),
            tokenizer_sha256=harness.tokenizer_fingerprint(tok),
            role="primary" if model_id == PRIMARY else "second curve", quant=quant)
        return mdl, tok

    print(f"generating on {GPU_NAME}")

## 5. Generating with a token budget

`run_at_budget` runs one problem at one budget:

1. **Cap.** Generate up to `budget` tokens. If the model ends its turn before the limit, the chain
   is finished and nothing else happens.
2. **Force an answer**, only if it hit the limit. Append `Final answer: \boxed{` and let the model
   write up to 32 more tokens, keeping its answer up to the brace that closes it.

The record keeps the two parts apart. `cut_index` marks where the model's own text ends and
`answer_start` where its forced answer begins. Natural-stop grading reads only the text before
`cut_index`, and the website draws the cut at exactly that point. Each token is stored as a string
(`tokens`), so the page can replay a chain at the token boundaries the model actually produced.

In [ ]:
FORCE_STEM = "\n\nFinal answer: \\boxed{"


def run_at_budget(question, budget, mdl, tok, stop_ids):
    chat = tok.apply_chat_template([{"role": "user", "content": render_prompt(question)}],
                                   tokenize=False, add_generation_prompt=True)
    prompt_ids = tok(chat, add_special_tokens=False).input_ids

    # 1. Cap. Getting back fewer tokens than the budget means the model stopped on its own.
    own_ids = harness.continue_ids(prompt_ids, mdl, tok, budget)
    hit_cap = len(own_ids) >= budget
    while own_ids and own_ids[-1] in stop_ids:      # the end-of-turn token is not text
        own_ids.pop()
    own = harness.incremental_token_strings(own_ids, tok)
    if not hit_cap:
        return dict(trace="".join(own), tokens=own, cut_index=None, answer_start=None,
                    truncated=False, forced=False, forced_suffix="", answer_closed=None,
                    own_tokens=len(own_ids), answer_tokens=0, tokens_generated=len(own_ids))

    # 2. Force an answer.
    stem_ids = tok(FORCE_STEM, add_special_tokens=False).input_ids
    stem = harness.incremental_token_strings(stem_ids, tok)
    assert "".join(stem) == FORCE_STEM, "the forcing text does not survive tokenisation"
    ans_ids = harness.continue_ids(prompt_ids + own_ids + stem_ids, mdl, tok, ANSWER_ALLOWANCE)
    while ans_ids and ans_ids[-1] in stop_ids:
        ans_ids.pop()
    answer, closed = harness.cut_at_boxed_close(harness.incremental_token_strings(ans_ids, tok))
    tokens = own + stem + answer
    return dict(trace="".join(tokens), tokens=tokens, cut_index=len(own),
                answer_start=len(own) + len(stem), truncated=True, forced=True,
                forced_suffix=FORCE_STEM, answer_closed=closed,
                own_tokens=len(own_ids), answer_tokens=len(answer),
                tokens_generated=len(own_ids) + len(answer))   # our stem is not the model's cost

## 6. Running every budget

Each model works through the budgets from smallest to largest, one problem at a time.

**A shortcut that changes nothing.** If a model finished a problem on its own at a smaller budget,
it would write exactly the same chain at any larger budget, because greedy decoding is
deterministic. That record is reused instead of generated again, and marked with `carried_from`.
Section 8 regenerates one reused record and checks that it comes out identical.

In [ ]:
def reusable(model_id, budget, problem_id):
    """A chain this model already finished on its own at a smaller budget, if any."""
    for smaller in reversed([b for b in BUDGETS if b < budget]):
        prev = STORE.done.get((model_id, smaller, problem_id))
        if prev and not prev["truncated"]:
            return prev
    return None


def sweep(model_id, mdl, tok):
    stop_ids = harness.generation_stop_ids(mdl)
    results[model_id] = {}
    for b in BUDGETS:                      # ascending, so a reusable chain always exists already
        for row in harness.progress(items, desc=f"{model_id.split('/')[-1]} @ {b}", unit="problem"):
            if (model_id, b, row["unique_id"]) in STORE.done:
                continue                   # already on disk from an earlier, interrupted run
            prev = reusable(model_id, b, row["unique_id"])
            if prev is not None:
                STORE.append(dict(prev, budget=b, carried_from=prev.get("carried_from") or prev["budget"]))
                continue
            g = run_at_budget(row["problem"], b, mdl, tok, stop_ids)
            answer = extract_final_candidate(g["trace"])
            STORE.append(dict(
                model_id=model_id, problem_id=row["unique_id"], level=int(row["level"]),
                subject=row["subject"], question=row["problem"], ground_truth=row["answer"],
                budget=b, think_closed=None, extracted_answer=answer,
                correct=bool(grade_answer(answer, row["answer"])), carried_from=None, **g))
        results[model_id][b] = [STORE.done[(model_id, b, r["unique_id"])] for r in items]


def spot_check_reuse(model_id, mdl, tok):
    """Regenerate one reused record directly and confirm it is identical."""
    r = next((r for b in BUDGETS for r in results[model_id][b] if r.get("carried_from")), None)
    if r is None:
        return None
    again = run_at_budget(r["question"], r["budget"], mdl, tok, harness.generation_stop_ids(mdl))
    check = dict(problem_id=r["problem_id"], budget=r["budget"],
                 carried_from=r["carried_from"], identical=again["trace"] == r["trace"])
    assert check["identical"], "a reused record does not reproduce when generated directly"
    return check


if GENERATES:
    CARRY_CHECK = None
    for model_id, quant in MODELS:         # one model in memory at a time
        mdl, tok = load(model_id, quant)
        sweep(model_id, mdl, tok)
        if model_id == PRIMARY:
            CARRY_CHECK = spot_check_reuse(model_id, mdl, tok)
        del mdl
        harness.free_memory()
else:
    CARRY_CHECK = RUN_MANIFEST["integrity"]["carry_forward_spot_check"]

# Same order everywhere, and a natural-stop grade on every record.
SWEPT = [model_id for model_id, _ in MODELS if model_id in results]
for model_id in SWEPT:
    for b in BUDGETS:
        recs = sorted(results[model_id][b], key=lambda r: ORDER[r["problem_id"]])
        assert len(recs) == len(items), f"{model_id} at {b} tokens has {len(recs)} of {len(items)} problems"
        for r in recs:
            r["natural_correct"] = natural_stop_correct(r)
        results[model_id][b] = recs

print(f"{'model':<24}" + "".join(f"{b:>8}" for b in BUDGETS) + "   problems solved, natural stop")
for model_id in SWEPT:
    print(f"{model_id.split('/')[-1]:<24}"
          + "".join(f"{sum(r['natural_correct'] for r in results[model_id][b]):>8}" for b in BUDGETS))

## 7. One problem, every budget

What the records look like for a single problem: how much the model wrote before the cap, whether
it was cut off, and how the chain is graded under natural stop and with a forced answer.

In [ ]:
def show_trace(r, width=94, head=4, tail=2):
    """The model's own words, where the budget ran out, and (if forced) what it committed to."""
    own = "".join(r["tokens"][: r["cut_index"]]) if r["forced"] else r["trace"]
    lines = textwrap.wrap(own.strip().replace("\n", " "), width) or ["(nothing generated)"]
    if len(lines) > head + tail:
        lines = lines[:head] + [f"... [{len(lines) - head - tail} lines] ..."] + lines[-tail:]
    out = [f"  {line}" for line in lines]
    if r["forced"]:
        out[-1] += " |cut|"
        forced = "".join(r["tokens"][r["answer_start"]:])
        out.append(f"  >> forced: Final answer: \\boxed{{{forced}")
    return "\n".join(out)


example = items[0]
print(textwrap.fill("Q: " + example["problem"], 96))
print("reference answer:", example["answer"])
for b in BUDGETS:
    r = next(r for r in results[PRIMARY][b] if r["problem_id"] == example["unique_id"])
    status = "cut off" if r["truncated"] else "finished"
    print(f"\n--- {b} tokens: wrote {r['own_tokens']}, {status}   "
          f"natural stop: {'correct' if r['natural_correct'] else 'wrong'}   "
          f"forced answer: {'correct' if r['correct'] else 'wrong'}")
    print(show_trace(r))

## 8. Checks on the run

Before drawing anything, six properties the figures depend on:

1. Every chain can be rebuilt exactly from its stored tokens (the website replays them).
2. No chain wrote more of its own tokens than its budget allowed (the x-axis means what it says).
3. A chain cut off at a small budget is exactly the beginning of the same problem's longer chain.
   This is what makes figure 1's greyed-out "what it would have written next" honest.
4. A reused record is identical to one generated directly (checked in section 6).
5. Forced answers almost always had room to close their brace.
6. How often the cap cut chains off at each budget. At the top budget it should rarely bind;
   if it did, the end of the curve would still be a cut rather than a finished chain.

In [ ]:
all_records = [r for model_id in SWEPT for b in BUDGETS for r in results[model_id][b]]
forced = [r for r in all_records if r["forced"]]
reused = [r for r in all_records if r.get("carried_from")]

assert all("".join(r["tokens"]) == r["trace"] for r in all_records), "1. a chain does not rebuild"
assert all(r["own_tokens"] <= r["budget"] for r in all_records), "2. a chain overran its budget"

# 3. cut chains are prefixes of the longest chain for the same problem
comparisons = []
for model_id in SWEPT:
    longest = {r["problem_id"]: r for r in results[model_id][MAX_BUDGET]}
    for b in BUDGETS[:-1]:
        for r in results[model_id][b]:
            if not r["truncated"]:
                continue                   # reused records would just be compared with themselves
            ref = longest[r["problem_id"]]
            own = r["tokens"][: r["cut_index"]]
            ref_own = ref["tokens"][: ref["cut_index"]] if ref["forced"] else ref["tokens"]
            comparisons.append(harness.prefix_len(own, ref_own) == len(own))
prefix_exact = mean(comparisons)
unclosed = sum(r["answer_closed"] is False for r in forced) / max(len(forced), 1)

INTEGRITY = dict(n_records=len(all_records), n_forced=len(forced), unclosed_forced_rate=unclosed,
                 prefix_exact_rate=prefix_exact, n_prefix_comparisons=len(comparisons),
                 n_carried_forward=len(reused), carry_forward_spot_check=CARRY_CHECK)

print(f"1-2. {len(all_records):,} records rebuild from their tokens and stay within budget")
print(f"3.   cut chains that are exact prefixes of the longer chain: {prefix_exact:.1%} of {len(comparisons)}")
print(f"4.   reused records: {len(reused)}; spot check: "
      + ("none needed" if CARRY_CHECK is None else
         f"{CARRY_CHECK['problem_id']} at {CARRY_CHECK['budget']} -> "
         f"{'identical' if CARRY_CHECK['identical'] else 'DIFFERENT'}"))
print(f"5.   forced answers that closed their brace: {1 - unclosed:.1%} of {len(forced)}")
print("6.   share of chains cut off by the cap:")
print(f"     {'budget':>8}" + "".join(f"{m.split('/')[-1]:>24}" for m in SWEPT))
for b in BUDGETS:
    print(f"     {b:>8}" + "".join(f"{mean([r['truncated'] for r in results[m][b]]):>24.0%}" for m in SWEPT))

# The top-of-grid check, on the first 30 problems for the primary model.
top = results[PRIMARY][MAX_BUDGET][:min(30, len(items))]
top_cut = mean([r["truncated"] for r in top])
PILOT = dict(n=len(top), budget=MAX_BUDGET, accuracy=mean([r["correct"] for r in top]),
             mean_own_tokens=mean([r["own_tokens"] for r in top]), truncation_rate=top_cut,
             verdict="grid brackets the model" if top_cut <= 0.10 else "TOP OF GRID STILL BINDS",
             advice="proceed" if top_cut <= 0.10 else "add the next doubling to BUDGETS")
print(f"\ntop budget ({MAX_BUDGET}): {top_cut:.0%} of the first {len(top)} chains still cut off -> {PILOT['verdict']}")

if RUN_MODE == "REPLAY":
    published = {k: v for k, v in RUN_MANIFEST["integrity"].items() if k != "carry_forward_spot_check"}
    recomputed = {k: v for k, v in INTEGRITY.items() if k != "carry_forward_spot_check"}
    assert recomputed == published, f"checks differ from the published run: {recomputed} vs {published}"
    print("the checks match the ones recorded by the run that generated these chains")

# Results

## 9. Figure 1: one problem at every budget

The website's first figure shows one problem cut off at whichever budget the reader picks. The
example is chosen from the run, not generated for the figure: a problem the primary model gets
wrong at a small budget and right at a larger one, where the budget just below is a genuine cut,
preferring problems that turn correct near the middle of the grid and have the shortest chains.

Past the cut, the website greys out what the model *would* have written. That text is the same
problem's chain at the largest budget (`reference_tokens`). Because decoding is greedy, the cut
chain is exactly its beginning (check 3), so the greyed text is real model output, not a
reconstruction.

The choice below still uses forced-answer grades, which is what the page currently draws. The
natural-stop grades are stored alongside each budget (`natural_correct`).

In [ ]:
def by_problem(model_id, key="correct"):
    """problem_id -> {budget: record}, in problem order."""
    out = {}
    for b in BUDGETS:
        for r in results[model_id][b]:
            out.setdefault(r["problem_id"], {})[b] = r
    return out


def first_correct(row, key="correct"):
    return next((b for b in BUDGETS if row[b][key]), None)


PRIMARY_BY_PROBLEM = by_problem(PRIMARY)

candidates = []
for pid, row in PRIMARY_BY_PROBLEM.items():
    flip = first_correct(row)
    if flip is None or flip == BUDGETS[0]:
        continue                                        # needs a visible "before"
    if not row[BUDGETS[BUDGETS.index(flip) - 1]]["truncated"]:
        continue                                        # the budget below must show a real cut
    distance_from_middle = abs(BUDGETS.index(flip) - (len(BUDGETS) - 1) / 2)
    candidates.append((distance_from_middle, len(row[MAX_BUDGET]["tokens"]), pid, flip))
candidates.sort()

SPECIMEN_IS_FALLBACK = not candidates
if candidates:
    _, _, SPECIMEN_ID, SPECIMEN_FLIP = candidates[0]
else:                                                   # flagged; the website refuses a fallback
    SPECIMEN_ID = min(PRIMARY_BY_PROBLEM, key=lambda pid: len(PRIMARY_BY_PROBLEM[pid][MAX_BUDGET]["tokens"]))
    SPECIMEN_FLIP = first_correct(PRIMARY_BY_PROBLEM[SPECIMEN_ID])

specimen = PRIMARY_BY_PROBLEM[SPECIMEN_ID]
print(f"{len(candidates)} candidate problems; chosen: {SPECIMEN_ID}, turns correct at {SPECIMEN_FLIP} tokens")
print(textwrap.fill("Q: " + specimen[MAX_BUDGET]["question"], 96))
print(f"{'budget':>8}{'wrote':>8}{'cut off':>9}{'natural stop':>14}{'forced':>9}")
for b in BUDGETS:
    r = specimen[b]
    print(f"{b:>8}{r['own_tokens']:>8}{'yes' if r['truncated'] else 'no':>9}"
          f"{'correct' if r['natural_correct'] else 'wrong':>14}{'correct' if r['correct'] else 'wrong':>9}")

## 10. Figure 2: accuracy against the token budget

Share of the 100 problems each model solves at each budget, under natural stop (solid) and with a
forced answer (dashed). Where the two lines meet, almost no chain is being cut off any more.

The table under the plot reports what each doubling of the budget bought, and the budget past which
no later doubling buys two points or more (the "bend"). A dip is kept, not smoothed away.

In [ ]:
def curve_for(model_id):
    pts = []
    for b in BUDGETS:
        rs = results[model_id][b]
        pts.append(dict(
            budget=b, n=len(rs),
            n_correct=sum(r["correct"] for r in rs), accuracy=mean([r["correct"] for r in rs]),
            mean_tokens=mean([r["tokens_generated"] for r in rs]),
            total_tokens=sum(r["tokens_generated"] for r in rs),
            truncation_rate=mean([r["truncated"] for r in rs]),
            forced_rate=mean([r["forced"] for r in rs]),
            natural_n_correct=sum(r["natural_correct"] for r in rs),
            natural_accuracy=mean([r["natural_correct"] for r in rs])))
    return pts


def bend(pts, key):
    """What each doubling bought, and where the curve stops paying (no later gain of 2+ points)."""
    gains = [dict(from_budget=a["budget"], to_budget=b["budget"], gain_pts=(b[key] - a[key]) * 100)
             for a, b in zip(pts, pts[1:])]
    knee = next((g["from_budget"] for i, g in enumerate(gains)
                 if all(h["gain_pts"] < 2.0 for h in gains[i:])), None)
    top_ = max(p[key] for p in pts)
    return dict(gains=gains, monotone=all(g["gain_pts"] >= 0 for g in gains), knee_budget=knee,
                ceiling=top_, floor=pts[0][key], total_range_pts=(top_ - pts[0][key]) * 100)


CURVES = {m: curve_for(m) for m in SWEPT}
BEND = {m: bend(CURVES[m], "accuracy") for m in SWEPT}
BEND_NATURAL = {m: bend(CURVES[m], "natural_accuracy") for m in SWEPT}

fig, ax = plt.subplots(figsize=(8, 4.2))
for (model_id, pts), colour in zip(CURVES.items(), ["#3b4a6b", "#a2704a", "#5c7a5c"]):
    label = model_id.split("/")[-1]
    ax.plot(BUDGETS, [p["natural_accuracy"] * 100 for p in pts], marker="o", ms=4, lw=2, color=colour, label=label)
    ax.plot(BUDGETS, [p["accuracy"] * 100 for p in pts], ls="--", lw=1, color=colour, alpha=0.7)
ax.set_xscale("log", base=2)
ax.set_xticks(BUDGETS)
ax.set_xticklabels([str(b) for b in BUDGETS])
ax.set_xlabel("token budget (each step doubles it)")
ax.set_ylabel("problems solved (%)")
ax.set_ylim(0, 100)
ax.grid(axis="y", alpha=0.25)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="upper left")
ax.set_title("solid: natural stop    dashed: forced answer", fontsize=9, loc="left", color="#555")
fig.tight_layout()
fig.savefig(OUT_DIR / "figure_2_curve.png", dpi=160)
plt.show()

for model_id in SWEPT:
    n, f = BEND_NATURAL[model_id], BEND[model_id]
    print(f"\n{model_id.split('/')[-1]}   gain per doubling (natural stop / forced answer)")
    for gn, gf in zip(n["gains"], f["gains"]):
        print(f"  {gn['from_budget']:>5} -> {gn['to_budget']:<5} {gn['gain_pts']:>+6.1f} / {gf['gain_pts']:>+6.1f} points")
    print(f"  stops paying after: {n['knee_budget'] or 'still rising'} / {f['knee_budget'] or 'still rising'}")

## 11. Figure 3: when each problem first comes right

One row per problem, coloured from the budget at which it is first graded correct. Each problem
behaves like a threshold, not a slope: below some budget it is wrong, above it right. Figure 2's
smooth climb is the average of many such steps landing in different places. A problem that is right
at one budget and wrong at a larger one (a regression) is marked, not hidden.

Under natural stop a problem usually turns correct at the first budget large enough for its chain
to finish. The plot shows natural stop; the forced-answer version is stored too, because the page
currently draws it.

In [ ]:
def flip_summary(rows_by_problem, key):
    items_ = []
    for pid, row in rows_by_problem.items():
        verdicts = [bool(row[b][key]) for b in BUDGETS]
        first = next((BUDGETS[i] for i, v in enumerate(verdicts) if v), None)
        items_.append(dict(pid=pid, verdicts=verdicts, first=first,
                           stays=all(verdicts[BUDGETS.index(first):]) if first else True))
    return items_


natural = {d["pid"]: d for d in flip_summary(PRIMARY_BY_PROBLEM, "natural_correct")}
flip_items = []
for d in flip_summary(PRIMARY_BY_PROBLEM, "correct"):
    top_rec = PRIMARY_BY_PROBLEM[d["pid"]][MAX_BUDGET]
    flip_items.append(dict(
        problem_id=d["pid"], level=top_rec["level"], subject=top_rec["subject"],
        correct_by_budget=d["verdicts"], first_correct_budget=d["first"], monotone=d["stays"],
        tokens_at_max=top_rec["tokens_generated"], truncated_at_max=top_rec["truncated"],
        natural_correct_by_budget=natural[d["pid"]]["verdicts"],
        natural_first_correct_budget=natural[d["pid"]]["first"]))

# never-solved problems last, then by the budget they turn correct at, then by chain length
flip_items.sort(key=lambda d: (d["first_correct_budget"] is None,
                               BUDGETS.index(d["first_correct_budget"]) if d["first_correct_budget"] else 0,
                               d["tokens_at_max"]))


def strip_stats(prefix):
    first = f"{prefix}first_correct_budget"
    verdicts = f"{prefix}correct_by_budget"
    never = [d for d in flip_items if d[first] is None]
    regress = [d for d in flip_items if d[first] and not all(d[verdicts][BUDGETS.index(d[first]):])]
    return dict(histogram={str(b): sum(d[first] == b for d in flip_items) for b in BUDGETS},
                never_flip_count=len(never), never_flip_share=len(never) / len(flip_items),
                n_regressions=len(regress))


STRIP, STRIP_NATURAL = strip_stats(""), strip_stats("natural_")

order = sorted(flip_items, key=lambda d: (d["natural_first_correct_budget"] is None,
                                          BUDGETS.index(d["natural_first_correct_budget"])
                                          if d["natural_first_correct_budget"] else 0, d["tokens_at_max"]))
fig, ax = plt.subplots(figsize=(7, 5))
ax.imshow(np.array([d["natural_correct_by_budget"] for d in order], dtype=int), aspect="auto",
          interpolation="nearest", cmap=ListedColormap(["#e8e6e1", "#3b4a6b"]), vmin=0, vmax=1)
ax.set_xticks(range(len(BUDGETS)))
ax.set_xticklabels([str(b) for b in BUDGETS])
ax.set_yticks([])
ax.set_xlabel("token budget")
ax.set_ylabel(f"problems (n={len(flip_items)}), sorted by when they first come right")
ax.set_title(f"{PRIMARY.split('/')[-1]}, natural stop", fontsize=9, loc="left", color="#555")
fig.tight_layout()
fig.savefig(OUT_DIR / "figure_3_flip_strip.png", dpi=160)
plt.show()

print(f"{'first correct at':>18}{'natural stop':>14}{'forced':>9}")
for b in BUDGETS:
    print(f"{b:>18}{STRIP_NATURAL['histogram'][str(b)]:>14}{STRIP['histogram'][str(b)]:>9}")
print(f"{'never':>18}{STRIP_NATURAL['never_flip_count']:>14}{STRIP['never_flip_count']:>9}")
print(f"regressions (right, then wrong at a larger budget): "
      f"{STRIP_NATURAL['n_regressions']} natural stop, {STRIP['n_regressions']} forced")

# Appendices

## A. Asking for a short answer instead of imposing a cap

The same primary model, the first 20 problems, and a polite request at the end of the prompt
("Keep your reasoning to one or two sentences, then give the final answer.") in place of a hard
limit. Generation runs up to the top budget, so the question is only how long the chains are, not
how accurate: the two arms' accuracies are not comparable, because one was capped and one was not.

The website quotes one number from this: the share of polite-request chains that ran past 512
tokens anyway.

In [ ]:
SOFT_BUDGET = BUDGETS[len(BUDGETS) // 2]        # the nominal budget, a middle step of the grid
SOFT_REQUEST = "\n\nKeep your reasoning to one or two sentences, then give the final answer."

if GENERATES and RUN_ASKED_NICELY:
    mdl, tok = load(PRIMARY, dict(MODELS)[PRIMARY])
    stop_ids = harness.generation_stop_ids(mdl)
    for row in harness.progress(items[:20], desc="asked nicely", unit="problem"):
        key = ("soft", SOFT_BUDGET, row["unique_id"])
        if key not in STORE.done:
            g = run_at_budget(row["problem"] + SOFT_REQUEST, MAX_BUDGET, mdl, tok, stop_ids)
            answer = extract_final_candidate(g["trace"])
            STORE.append(dict(model_id="soft", problem_id=row["unique_id"], level=int(row["level"]),
                              subject=row["subject"], question=row["problem"], ground_truth=row["answer"],
                              budget=SOFT_BUDGET, arm="asked_nicely", request=SOFT_REQUEST.strip(),
                              think_closed=None, extracted_answer=answer,
                              correct=bool(grade_answer(answer, row["answer"])), carried_from=None, **g))
        asked_nicely.append(STORE.done[key])
    del mdl
    harness.free_memory()

INSET = None
if asked_nicely:
    asked_nicely.sort(key=lambda r: ORDER[r["problem_id"]])
    SOFT_BUDGET = asked_nicely[0]["budget"]
    ids = {r["problem_id"] for r in asked_nicely}
    capped = [r for r in results[PRIMARY][SOFT_BUDGET] if r["problem_id"] in ids]

    def summary(rs, over_budget_rate, measured_at_budget):
        return dict(mean_own_tokens=mean([r["own_tokens"] for r in rs]),
                    median_own_tokens=harness.median_midpoint([r["own_tokens"] for r in rs]),
                    over_budget_rate=over_budget_rate, measured_at_budget=measured_at_budget,
                    accuracy=mean([r["correct"] for r in rs]))

    INSET = dict(nominal_budget=SOFT_BUDGET, n=len(asked_nicely), request=asked_nicely[0].get("request"),
                 soft=summary(asked_nicely, mean([r["own_tokens"] > SOFT_BUDGET for r in asked_nicely]), MAX_BUDGET),
                 hard=summary(capped, 0.0, SOFT_BUDGET))
    INSET["overshoot_multiple"] = INSET["soft"]["mean_own_tokens"] / SOFT_BUDGET
    print(f"{'':<14}{'mean tokens':>12}{'median':>9}{f'over {SOFT_BUDGET}':>10}")
    for name, k in (("asked nicely", "soft"), ("capped", "hard")):
        d = INSET[k]
        print(f"{name:<14}{d['mean_own_tokens']:>12,.0f}{d['median_own_tokens']:>9,.0f}{d['over_budget_rate']:>10.0%}")
else:
    print("no asked-nicely records in this run")

## B. The opposite of a cap: telling the model to keep going

Everything above takes room away. This arm refuses to let the model stop: when it tries to end its
turn, the end-of-turn token is dropped, the word `Wait` is appended to what it has written, and
generation continues. Repeat up to three times and grade after each. This is the other half of
budget forcing (Muennighoff et al., 2025).

These models have no separate thinking section, so the word lands after a finished answer, not in
the middle of deliberating. Muennighoff et al. also fine-tuned their model on reasoning traces
first; these models had no such training, so this is the word without the training.

`k = 0` is the model left alone with the top budget, so its chains must match the main sweep's.
If the model is still writing when a step's allowance runs out, its answer is forced exactly as in
section 5. Those forced records can only cost the intervention marks, so the paired counts are
reported twice: over all problems, and over the "clean" ones where no allowance ran out.

In [ ]:
WAIT_TEXT = "\n" + WAIT_WORD


def run_with_waits(question, mdl, tok, stop_ids, n_waits):
    """One growing chain, sampled after each "Wait": record k is record k-1 plus our word
    plus whatever the model wrote next."""
    chat = tok.apply_chat_template([{"role": "user", "content": render_prompt(question)}],
                                   tokenize=False, add_generation_prompt=True)
    prompt_ids = tok(chat, add_special_tokens=False).input_ids
    wait_ids = tok(WAIT_TEXT, add_special_tokens=False).input_ids
    stem_ids = tok(FORCE_STEM, add_special_tokens=False).input_ids
    stem = harness.incremental_token_strings(stem_ids, tok)

    body, spans, out = [], [], []
    for k in range(n_waits + 1):
        room = MAX_BUDGET if k == 0 else WAIT_ALLOWANCE
        fresh = harness.continue_ids(prompt_ids + body, mdl, tok, room)
        ran_on = len(fresh) >= room                     # it never offered to stop
        while fresh and fresh[-1] in stop_ids:
            fresh.pop()
        body += fresh
        pieces = harness.incremental_token_strings(body, tok)
        own = len(body) - sum(e - s for s, e in spans)  # our injected words are not its cost
        if ran_on:                                      # force an answer, as in section 5
            ans = harness.continue_ids(prompt_ids + body + stem_ids, mdl, tok, ANSWER_ALLOWANCE)
            while ans and ans[-1] in stop_ids:
                ans.pop()
            kept, closed = harness.cut_at_boxed_close(harness.incremental_token_strings(ans, tok))
            toks = pieces + stem + kept
            rec = dict(trace="".join(toks), tokens=toks, cut_index=len(pieces),
                       answer_start=len(pieces) + len(stem), truncated=True, forced=True,
                       forced_suffix=FORCE_STEM, answer_closed=closed, answer_tokens=len(kept),
                       own_tokens=own, tokens_generated=own + len(kept))
        else:
            rec = dict(trace="".join(pieces), tokens=pieces, cut_index=None, answer_start=None,
                       truncated=False, forced=False, forced_suffix="", answer_closed=None,
                       answer_tokens=0, own_tokens=own, tokens_generated=own)
        rec.update(waits=k, wait_spans=[list(s) for s in spans], offered_a_stop=not ran_on)
        out.append(rec)
        if ran_on:
            break                                       # nothing left to suppress
        spans.append((len(body), len(body) + len(wait_ids)))
        body += wait_ids
    return out


if GENERATES and RUN_WAIT_ARM:
    # Resume from the wait records already on disk. They share traces.jsonl with the
    # budget sweep but are keyed by the number of waits, so they are read here directly.
    done = {}
    with TRACES_PATH.open(encoding="utf-8") as f:
        for r in (json.loads(line) for line in f if line.strip()):
            if r.get("run_id") == RUN_ID and r.get("arm") == "wait":
                done[(r["model_id"], r["waits"], r["problem_id"])] = r
    for model_id, quant in MODELS:
        mdl, tok = load(model_id, quant)
        stop_ids = harness.generation_stop_ids(mdl)
        for row in harness.progress(items, desc=f"waits {model_id.split('/')[-1]}", unit="problem"):
            if all((model_id, k, row["unique_id"]) in done for k in WAITS):
                continue
            snaps = {s["waits"]: s for s in run_with_waits(row["problem"], mdl, tok, stop_ids, max(WAITS))}
            last = None
            for k in WAITS:
                s, carried = (snaps[k], None) if k in snaps else (last, last["waits"])
                answer = extract_final_candidate(s["trace"])
                done[(model_id, k, row["unique_id"])] = STORE.append(dict(
                    model_id=model_id, problem_id=row["unique_id"], level=int(row["level"]),
                    subject=row["subject"], question=row["problem"], ground_truth=row["answer"],
                    arm="wait", budget=None, wait_word=WAIT_WORD, think_closed=None,
                    extracted_answer=answer, correct=bool(grade_answer(answer, row["answer"])),
                    carried_from=carried, **s))
                if k in snaps:
                    last = s
        wait_records[model_id] = [done[(model_id, k, r["unique_id"])] for r in items for k in WAITS]
        del mdl
        harness.free_memory()

for model_id in wait_records:
    wait_records[model_id].sort(key=lambda r: (ORDER[r["problem_id"]], r["waits"]))
print(sum(len(v) for v in wait_records.values()), "wait-arm records")

In [ ]:
def own_text(r):
    return "".join(r["tokens"][: r["cut_index"]]) if r["forced"] else r["trace"]


WAIT_CHECKS = {}
if wait_records:
    lead = PRIMARY if PRIMARY in wait_records else sorted(wait_records)[0]
    flat = [r for rs in wait_records.values() for r in rs]

    # k = 0 is the model left alone: it must match the main sweep at the top budget.
    uncapped = {r["problem_id"]: r for r in results[lead][MAX_BUDGET] if not r["truncated"]}
    zero = [r for r in wait_records[lead] if r["waits"] == 0 and r["problem_id"] in uncapped]
    same = sum(r["trace"] == uncapped[r["problem_id"]]["trace"] for r in zero)
    WAIT_CHECKS["k0_matches_uncapped"] = (same, len(zero))

    assert all("".join(r["tokens"]) == r["trace"] for r in flat), "a chain does not rebuild"
    breaks = 0                                          # each k extends the one before
    for rs in wait_records.values():
        by = {}
        for r in rs:
            by.setdefault(r["problem_id"], {})[r["waits"]] = r
        breaks += sum(not own_text(ks[b]).startswith(own_text(ks[a]))
                      for ks in by.values() for a, b in zip(WAITS, WAITS[1:]) if a in ks and b in ks)
    WAIT_CHECKS["prefix_breaks"] = breaks
    assert not breaks, "a wait-arm record does not extend the one before it"
    miscounted = [r for r in flat if r["own_tokens"] != len(r["tokens"][: r["cut_index"]] if r["forced"]
                                                             else r["tokens"]) - sum(e - s for s, e in r["wait_spans"])]
    WAIT_CHECKS["miscounted"] = len(miscounted)
    assert not miscounted, "an injected word was counted as the model's own output"
    WAIT_CHECKS["allowance_bound_at_kmax"] = {
        m: mean([r["forced"] for r in rs if r["waits"] == max(WAITS)]) for m, rs in wait_records.items()}

    print(f"k = 0 matches the main sweep: {same} of {len(zero)} chains identical")
    print("every record rebuilds, extends the one before it, and excludes our words from its token count")
    print("the 512-token allowance ran out at the last step for: "
          + ", ".join(f"{m.split('/')[-1]} {v:.0%}" for m, v in WAIT_CHECKS["allowance_bound_at_kmax"].items()))

In [ ]:
def same_answer(a, b):
    """Equivalent answers count as the same answer ("4" and "4.0")."""
    return a == b or (bool(a) and bool(b) and bool(grade_answer(a, b)))


WAIT_CURVES, WAIT_PAIRED, WAIT_VERDICT = {}, {}, {}
for model_id, rs in wait_records.items():
    by_k = {}
    for r in rs:
        by_k.setdefault(r["waits"], {})[r["problem_id"]] = r
    base = by_k[WAITS[0]]
    WAIT_CURVES[model_id] = []
    for k in WAITS:
        at_k = [r for r in rs if r["waits"] == k]
        moved = [not same_answer(r["extracted_answer"], base[r["problem_id"]]["extracted_answer"]) for r in at_k]
        WAIT_CURVES[model_id].append(dict(
            waits=k, n=len(at_k), churn=mean(moved) if moved else 0.0,
            n_correct=sum(r["correct"] for r in at_k), accuracy=mean([r["correct"] for r in at_k]),
            mean_tokens=mean([r["own_tokens"] for r in at_k]), total_tokens=sum(r["own_tokens"] for r in at_k),
            offered_a_stop=mean([r["offered_a_stop"] for r in at_k]),
            carried=mean([r["carried_from"] is not None for r in at_k])))

    # Problems fixed and broken against k = 0, with an exact McNemar test (as on page 5).
    WAIT_PAIRED[model_id] = []
    for k in WAITS[1:]:
        pids = sorted(set(base) & set(by_k[k]))
        fixed = [p for p in pids if by_k[k][p]["correct"] and not base[p]["correct"]]
        broken = [p for p in pids if base[p]["correct"] and not by_k[k][p]["correct"]]
        _, _, p_value = harness.mcnemar({q: by_k[k][q]["correct"] for q in pids}, {q: base[q]["correct"] for q in pids})
        clean = [p for p in pids if not by_k[k][p]["forced"] and not base[p]["forced"]
                 and by_k[k][p].get("carried_from") is None]
        c_fixed, c_broken, c_p = harness.mcnemar({q: by_k[k][q]["correct"] for q in clean},
                                                 {q: base[q]["correct"] for q in clean})
        WAIT_PAIRED[model_id].append(dict(waits=k, fixed=len(fixed), broken=len(broken), p=p_value, n=len(pids),
                                          fixed_ids=fixed, broken_ids=broken,
                                          clean=dict(n=len(clean), fixed=c_fixed, broken=c_broken, p=c_p,
                                                     excluded=len(pids) - len(clean))))

    pts, top_ = WAIT_CURVES[model_id], WAIT_PAIRED[model_id][-1]
    delta, churn = (pts[-1]["accuracy"] - pts[0]["accuracy"]) * 100, pts[-1]["churn"]
    verdict = ("the injections help" if delta >= 2 and top_["p"] < 0.05 else
               "the injections do nothing: it restates the answer it already had" if churn < 0.05 else
               "the injections hurt" if delta <= -2 and top_["p"] < 0.05 else
               "the injections churn without gaining")
    WAIT_VERDICT[model_id] = dict(delta_pts=delta, churn=churn, fixed=top_["fixed"], broken=top_["broken"],
                                  p=top_["p"], verdict=verdict)
    print(f"{model_id.split('/')[-1]:<24}{pts[0]['accuracy']:.0%} -> {pts[-1]['accuracy']:.0%} after "
          f"{WAITS[-1]} waits; {churn:.0%} of answers moved; {top_['fixed']} fixed, {top_['broken']} broken "
          f"(p = {top_['p']:.3f}): {verdict}")

if WAIT_CURVES:
    fig, ax = plt.subplots(figsize=(7, 3.8))
    for (model_id, pts), colour in zip(WAIT_CURVES.items(), ["#3b4a6b", "#a2704a", "#5c7a5c"]):
        ax.plot(WAITS, [p["accuracy"] * 100 for p in pts], marker="o", ms=4, lw=2, color=colour,
                label=model_id.split("/")[-1])
    ax.set_xticks(WAITS)
    ax.set_xlabel(f'times "{WAIT_WORD}" was appended')
    ax.set_ylabel("problems solved (%)")
    ax.set_ylim(0, 100)
    ax.grid(axis="y", alpha=0.25)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(OUT_DIR / "figure_4_waits.png", dpi=160)
    plt.show()

In [ ]:
# Two example chains for the website: one the extra thinking rescued, one it wrecked,
# each the shortest such chain in the run (so the page can show it whole).
WAIT_BY_PID = {}
WAIT_SPECIMENS = {"fixed": None, "broken": None, "is_fallback": {}}
if wait_records:
    lead = PRIMARY if PRIMARY in wait_records else sorted(wait_records)[0]
    for r in wait_records[lead]:
        WAIT_BY_PID.setdefault(r["problem_id"], {})[r["waits"]] = r

    def shortest(rescued):
        best = None
        for pid, by_k in WAIT_BY_PID.items():
            v = [by_k[k]["correct"] for k in WAITS]
            hit = (not v[0] and any(v[1:])) if rescued else (v[0] and not all(v))
            size = by_k[WAITS[-1]]["own_tokens"]
            if hit and (best is None or size < best[0]):
                best = (size, pid)
        return best[1] if best else None

    for kind, rescued in (("fixed", True), ("broken", False)):
        WAIT_SPECIMENS[kind] = shortest(rescued)
        WAIT_SPECIMENS["is_fallback"][kind] = WAIT_SPECIMENS[kind] is None
    print("rescued:", WAIT_SPECIMENS["fixed"], "   wrecked:", WAIT_SPECIMENS["broken"])

## C. Numbers handed to page 7

Page 7 spends the same total budget differently: instead of one chain of 2,048 tokens, 16 chains
of 128. These are the reference points it compares against.

In [ ]:
top_pt, bottom_pt = CURVES[PRIMARY][-1], CURVES[PRIMARY][0]
HANDOFF = dict(model_id=PRIMARY, max_budget=MAX_BUDGET, min_budget=BUDGETS[0],
               chains_at_min_budget=MAX_BUDGET // BUDGETS[0],
               accuracy_at_max=top_pt["accuracy"], accuracy_at_min=bottom_pt["accuracy"],
               natural_accuracy_at_max=top_pt["natural_accuracy"], natural_accuracy_at_min=bottom_pt["natural_accuracy"],
               total_tokens_at_max=top_pt["total_tokens"], mean_tokens_at_max=top_pt["mean_tokens"],
               never_flip_share=STRIP["never_flip_share"], never_flip_count=STRIP["never_flip_count"],
               n_items=len(flip_items))
print(f"one chain of {MAX_BUDGET} tokens: {top_pt['natural_accuracy']:.0%} solved (natural stop)")
print(f"one chain of {BUDGETS[0]} tokens: {bottom_pt['natural_accuracy']:.0%} solved (natural stop), "
      f"{bottom_pt['accuracy']:.0%} with a forced answer")

# Export for the website

`derived.json` is everything the website draws; the page runs no model. It bundles full chains
only for the problems figure 1 can show (its example plus 8 more spread across figure 3's rows);
every other chain stays in `traces.jsonl`, which ships as the download.

The website reads the forced-answer fields (`correct`, `accuracy`, `first_correct_budget`) today.
Every natural-stop number sits beside them under a `natural_` name.

In `REPLAY` mode the manifest is the one written by the run that generated the chains, so the file
still says how those chains were made. The replay adds a `replay` entry recording which notebook
rebuilt the numbers and when.

In [ ]:
BUNDLED = 8
spread = sorted({round(i * (len(flip_items) - 1) / (BUNDLED - 1)) for i in range(BUNDLED)})
bundled_ids = [flip_items[i]["problem_id"] for i in spread]
if SPECIMEN_ID not in bundled_ids:
    bundled_ids.insert(0, SPECIMEN_ID)

RUN_FIELDS = ("budget", "trace", "tokens", "cut_index", "answer_start", "truncated", "forced",
              "forced_suffix", "answer_closed", "extracted_answer", "correct", "own_tokens",
              "answer_tokens", "tokens_generated", "natural_correct")


def bundle_row(pid):
    row, ref = PRIMARY_BY_PROBLEM[pid], PRIMARY_BY_PROBLEM[pid][MAX_BUDGET]
    return dict(problem_id=pid, level=ref["level"], subject=ref["subject"], question=ref["question"],
                ground_truth=ref["ground_truth"], first_correct_budget=first_correct(row),
                natural_first_correct_budget=first_correct(row, "natural_correct"),
                reference_tokens=ref["tokens"][: ref["cut_index"]] if ref["forced"] else ref["tokens"],
                reference_truncated=ref["truncated"],
                by_budget=[{k: row[b][k] for k in RUN_FIELDS} for b in BUDGETS])


WAIT_FIELDS = ("waits", "trace", "tokens", "wait_spans", "cut_index", "answer_start", "truncated", "forced",
               "forced_suffix", "answer_closed", "offered_a_stop", "extracted_answer", "correct",
               "own_tokens", "answer_tokens", "tokens_generated", "carried_from")


def wait_bundle_row(pid):
    by_k, ref = WAIT_BY_PID[pid], WAIT_BY_PID[pid][WAITS[-1]]
    return dict(problem_id=pid, level=ref["level"], subject=ref["subject"], question=ref["question"],
                ground_truth=ref["ground_truth"],
                first_correct_waits=next((k for k in WAITS if by_k[k]["correct"]), None),
                by_waits=[{f: by_k[k][f] for f in WAIT_FIELDS} for k in WAITS])


if GENERATES:
    manifest = {
        "run_id": RUN_ID, "notebook": "page_06_thinking_longer.ipynb", "page": 6, "run_mode": RUN_MODE,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "notebook_git_sha": harness.git_sha(),
        "notebook_sha256": harness.file_sha256("page_06_thinking_longer.ipynb"),
        "models": [model_meta[m] for m in SWEPT],
        "dtype_requested": str(DTYPE), "replayed": False,
        "decoding": {"strategy": "greedy", "do_sample": False, "temperature": None, "top_p": None, "top_k": None},
        "tf32_matmul": bool(torch.backends.cuda.matmul.allow_tf32),
        "budgets": BUDGETS,
        "waits": {"word": WAIT_WORD, "grid": WAITS, "allowance": WAIT_ALLOWANCE, "injection": WAIT_TEXT,
                  "where": "appended when the model tries to end its turn; these models have no "
                           "thinking section, so that is after a finished answer"},
        "forcing": {"stem": FORCE_STEM, "answer_allowance": ANSWER_ALLOWANCE},
        "items": {"source": "MATH-500 (ch03 copy)", "dataset": "math500_l45", "n": len(items),
                  "item_set_sha256": ITEM_SET_HASH, "seed": SEED},
        "grading": {"extract": "reasoning_from_scratch.ch03.extract_final_candidate",
                    "grade": "reasoning_from_scratch.ch03.grade_answer"},
        "prompt_template": render_prompt("{question}"),
        "chat_template": tok.apply_chat_template([{"role": "user", "content": render_prompt("What is 2+2?")}],
                                                 tokenize=False, add_generation_prompt=True),
        "versions": {"torch": torch.__version__, "transformers": __import__("transformers").__version__,
                     "device": str(device), "gpu": GPU_NAME},
        "integrity": INTEGRITY,
    }
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
else:
    manifest = RUN_MANIFEST

derived = {
    "manifest": manifest,
    "budgets": BUDGETS,
    "procedure": (f"Cap generation at N tokens; if the chain is cut off, append \"Final answer: \\boxed{{\" "
                  f"and allow {ANSWER_ALLOWANCE} more tokens; grade with MATH-500's reference answer. Repeat "
                  f"for N in {{{', '.join(map(str, BUDGETS))}}} over {len(items)} problems from levels 4-5, "
                  f"greedy throughout."),
    "procedure_natural": (f"Cap generation at N tokens; a chain that is cut off before writing a boxed "
                          f"answer counts as wrong; grade with MATH-500's reference answer. Repeat for N in "
                          f"{{{', '.join(map(str, BUDGETS))}}} over {len(items)} problems from levels 4-5, "
                          f"greedy throughout."),
    "pilot": PILOT,
    "figure_1": {"specimen_id": SPECIMEN_ID, "specimen_is_fallback": SPECIMEN_IS_FALLBACK,
                 "flip_budget": SPECIMEN_FLIP, "n_candidate_specimens": len(candidates),
                 "rows": [bundle_row(pid) for pid in bundled_ids], "inset": INSET},
    "figure_2": {"curves": [{"model_id": m, "label": model_meta[m]["label"], "params": model_meta[m]["params"],
                             "points": CURVES[m], **BEND[m], "natural": BEND_NATURAL[m]} for m in SWEPT]},
    "figure_3": {"model_id": PRIMARY, "items": flip_items, **STRIP, "clickable_row_ids": bundled_ids,
                 "natural": STRIP_NATURAL},
    "figure_wait": None if not WAIT_CURVES else {
        "waits": WAITS, "word": WAIT_WORD, "n_items": len(items), "item_set_sha256": ITEM_SET_HASH,
        "allowance": WAIT_ALLOWANCE, "budget_arm_n_items": len(items),
        "curves": [{"model_id": m, "label": model_meta[m]["label"], "params": model_meta[m]["params"],
                    "points": WAIT_CURVES[m], "paired": WAIT_PAIRED[m]} for m in WAIT_CURVES],
        "specimen_fixed": WAIT_SPECIMENS["fixed"], "specimen_broken": WAIT_SPECIMENS["broken"],
        "specimen_is_fallback": WAIT_SPECIMENS["is_fallback"],
        "rows": [wait_bundle_row(p) for p in (WAIT_SPECIMENS["fixed"], WAIT_SPECIMENS["broken"]) if p],
        "integrity": WAIT_CHECKS},
    "handoff": HANDOFF,
}
if not GENERATES:
    derived["replay"] = {"notebook_sha256": harness.file_sha256("page_06_thinking_longer.ipynb"),
                         "notebook_git_sha": harness.git_sha(),
                         "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")}

DERIVED_PATH.write_text(json.dumps(derived, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"derived.json   {DERIVED_PATH.stat().st_size / 1024:,.0f} KB   (bundled with the page)")
print(f"traces.jsonl   {TRACES_PATH.stat().st_size / 1024:,.0f} KB   (the download)")
print("To publish: copy out/page_06/derived.json to web/data/page-06-derived.json. The website's "
      "tests recompute these numbers and refuse a file that disagrees.")